# Gold Layer — Analytics Aggregations

Gold views:
- **daily_sales_fact**: Daily revenue, order count, avg order value
- **customer_metrics**: Lifetime value, total spend, days since last purchase
- **product_metrics**: Total revenue, units sold, avg price by product
- **category_trends**: 7-day moving average of revenue by category

In [1]:
%pip install pyspark==3.5.3

  Using cached pyspark-3.5.3-py2.py3-none-any.whl
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.0
    Can't uninstall 'pyspark'. No files were found to uninstall.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .appName("explore_gold") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://dataops-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataops-key") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataops-secret") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print(f"Spark {spark.version} ready")

Spark 3.5.0 ready


In [3]:
BASE = "s3a://delta-lake/gold"
tables = ["daily_sales_fact", "customer_metrics", "product_metrics", "category_trends"]
for t in tables:
    df = spark.read.format("delta").load(f"{BASE}/{t}")
    print(f"\n{'='*60}")
    print(f"{t.upper()}: {df.count()} rows \u00d7 {len(df.columns)} cols")
    df.printSchema()
    df.show(5, truncate=False)


DAILY_SALES_FACT: 100 rows × 13 cols
root
 |-- txn_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- transaction_count: long (nullable = true)
 |-- completed_count: long (nullable = true)
 |-- refunded_count: long (nullable = true)
 |-- _updated_at: timestamp (nullable = true)
 |-- _load_date: string (nullable = true)

+----------+-----------+-----------+-----------+--------------+--------+------------+----------+-----------------+---------------+--------------+--------------------------+----------+
|txn_date  |customer_id|product_id |category   |store_location|quantity|gross_amount|net_amount|transaction_count|completed_count|refunded_count|_updated_at               |_load_date|
+----------+-----------+-------

In [4]:
# Top 10 customers by lifetime value
df = spark.read.format("delta").load(f"{BASE}/customer_metrics")
print("Top 10 customers by lifetime value:")
df.orderBy("customer_lifetime_value", ascending=False).select(
    "customer_id", "total_transactions", "customer_lifetime_value", "first_purchase_date", "last_purchase_date"
).show(10, truncate=False)

Top 10 customers by lifetime value:
+-----------+------------------+-----------------------+-------------------+------------------+
|customer_id|total_transactions|customer_lifetime_value|first_purchase_date|last_purchase_date|
+-----------+------------------+-----------------------+-------------------+------------------+
|CUST_000011|4                 |4100.969999999999      |2026-04-26         |2026-06-23        |
|CUST_000060|4                 |3196.65                |2026-04-04         |2026-06-10        |
|CUST_000037|3                 |2986.09                |2026-04-01         |2026-05-09        |
|CUST_000009|3                 |2686.69                |2026-04-10         |2026-06-09        |
|CUST_000014|3                 |2676.7999999999997     |2026-05-12         |2026-06-14        |
|CUST_000072|4                 |2482.15                |2026-04-18         |2026-05-05        |
|CUST_000010|1                 |2169.45                |2026-06-20         |2026-06-20        |
|CUS

In [5]:
# Top products by revenue
df = spark.read.format("delta").load(f"{BASE}/product_metrics")
print("Top 10 products by revenue:")
df.orderBy("total_revenue", ascending=False).select(
    "product_id", "product_name", "category", "total_quantity_sold", "total_revenue", "last_sale_date"
).show(10, truncate=False)

Top 10 products by revenue:
+-----------+--------------------+-----------+-------------------+------------------+--------------+
|product_id |product_name        |category   |total_quantity_sold|total_revenue     |last_sale_date|
+-----------+--------------------+-----------+-------------------+------------------+--------------+
|PROD_000023|Food Item #23       |Food       |14                 |6197.799999999999 |2026-06-10    |
|PROD_000050|Sports Item #50     |Sports     |16                 |5175.200000000001 |2026-06-27    |
|PROD_000043|Beauty Item #43     |Beauty     |11                 |4772.79           |2026-06-27    |
|PROD_000022|Electronics Item #22|Electronics|12                 |4000.67           |2026-06-18    |
|PROD_000029|Food Item #29       |Food       |11                 |3984.8900000000003|2026-06-19    |
|PROD_000036|Beauty Item #36     |Beauty     |10                 |3183.35           |2026-06-10    |
|PROD_000017|Clothing Item #17   |Clothing   |8                

In [6]:
# Daily revenue trend
df = spark.read.format("delta").load(f"{BASE}/daily_sales_fact")
print("Daily sales trend:")
df.orderBy("txn_date").select(
    "txn_date", "gross_amount", "net_amount", "transaction_count"
).show(15, truncate=False)

Daily sales trend:
+----------+------------+----------+-----------------+
|txn_date  |gross_amount|net_amount|transaction_count|
+----------+------------+----------+-----------------+
|2026-04-01|825.94      |825.94    |1                |
|2026-04-02|0.0         |229.16    |1                |
|2026-04-03|2149.55     |2149.55   |1                |
|2026-04-03|1770.8      |1770.8    |1                |
|2026-04-03|45.05       |45.05     |1                |
|2026-04-03|151.1       |151.1     |1                |
|2026-04-04|1005.53     |1005.53   |1                |
|2026-04-05|760.17      |760.17    |1                |
|2026-04-05|744.84      |744.84    |1                |
|2026-04-06|1117.26     |1117.26   |1                |
|2026-04-07|54.9        |54.9      |1                |
|2026-04-09|1328.1      |1328.1    |1                |
|2026-04-10|320.13      |320.13    |1                |
|2026-04-10|108.04      |108.04    |1                |
|2026-04-12|0.0         |187.44    |1         

In [7]:
# Category trends with 7-day moving average
df = spark.read.format("delta").load(f"{BASE}/category_trends")
print("Category revenue trends (7-day moving avg):")
df.orderBy("category", "trend_date").show(20, truncate=False)

Category revenue trends (7-day moving avg):
+----------+--------+------------------+--------------+-----------------+--------------------------+----------+
|trend_date|category|daily_sales       |daily_quantity|moving_avg_7_days|_updated_at               |_load_date|
+----------+--------+------------------+--------------+-----------------+--------------------------+----------+
|2026-04-10|Beauty  |108.04            |2             |108.04           |2026-06-30 07:51:08.691335|2026-06-30|
|2026-04-17|Beauty  |395.59999999999997|3             |251.82           |2026-06-30 07:51:08.691335|2026-06-30|
|2026-04-21|Beauty  |70.05             |1             |191.23           |2026-06-30 07:51:08.691335|2026-06-30|
|2026-05-03|Beauty  |212.4             |4             |196.52           |2026-06-30 07:51:08.691335|2026-06-30|
|2026-05-06|Beauty  |1607.75           |5             |478.77           |2026-06-30 07:51:08.691335|2026-06-30|
|2026-05-09|Beauty  |1735.56           |4             |688.2

In [8]:
spark.stop()